QN.Calculate the percentage marks for each student.Eachsubject is of 100 marks.Create a result column by following the below conditions:

1.%marks greater than or equal to 70 then 'DISTINCTION'

2.%marks range between 60-69 then 'FIRST CLASS'

3.%marks range betwen 50-59 then 'SECOND CLASS'

4.%marks range between 40-49 the 'THIRD CLASS'

5.%marks less than or equal to 39 the 'FAIL'


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, round, when

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("StudentResultCalculation") \
    .getOrCreate()

# Sample data1 (Student ID and Name)
data1 = [
    (1, "Sam"),
    (2, "Dany"),
    (3, "Joey"),
    (4, "Lara"),
    (5, "Helen")
]

# Sample data2 (Student ID, Subject, Marks)
data2 = [
    (1, "SUB1", 90),
    (1, "SUB2", 100),
    (2, "SUB1", 70),
    (2, "SUB2", 60),
    (3, "SUB1", 30),
    (3, "SUB2", 20),
    (4, "SUB1", 50),
    (4, "SUB2", 50),
    (5, "SUB1", 45),
    (5, "SUB2", 45)
]

# Define schemas
schema1 = ["Id", "Name"]
schema2 = ["Id", "Subject", "Mark"]

# Create DataFrames
df1 = spark.createDataFrame(data1, schema=schema1)
df2 = spark.createDataFrame(data2, schema=schema2)

# Join the dataframes based on "Id"
joined_df = df1.join(df2, on="Id", how="inner")

# Calculate total marks for each student
total_marks_df = joined_df.groupBy("Id", "Name") \
    .agg(sum("Mark").alias("Total_Marks"))

# Calculate the percentage (out of 200 marks, since each subject is out of 100)
result_df = total_marks_df.withColumn("Percentage", (col("Total_Marks") / 200) * 100)

# Categorize students based on their percentage
result_df = result_df.withColumn(
    "Result",
    when(col("Percentage") >= 70, "DISTINCTION")
    .when((col("Percentage") >= 60) & (col("Percentage") < 70), "FIRST CLASS")
    .when((col("Percentage") >= 50) & (col("Percentage") < 60), "SECOND CLASS")
    .when((col("Percentage") >= 40) & (col("Percentage") < 50), "THIRD CLASS")
    .otherwise("FAIL")
)

# Show the final result
result_df.show()


+---+-----+-----------+----------+------------+
| Id| Name|Total_Marks|Percentage|      Result|
+---+-----+-----------+----------+------------+
|  1|  Sam|        190|      95.0| DISTINCTION|
|  2| Dany|        130|      65.0| FIRST CLASS|
|  3| Joey|         50|      25.0|        FAIL|
|  4| Lara|        100|      50.0|SECOND CLASS|
|  5|Helen|         90|      45.0| THIRD CLASS|
+---+-----+-----------+----------+------------+

